In [5]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
start = time.time()
movies_data= pd.read_csv("movies.csv")
print("FILE LOADED | This operation took", time.time() - start, "seconds")

FILE LOADED | This operation took 0.10199713706970215 seconds


In [6]:
print(movies_data.columns.values)

['MOVIES' 'YEAR' 'GENRE' 'RATING' 'ONE-LINE' 'STARS' 'VOTES' 'RunTime'
 'Gross']


In [7]:
# We have drpped unneccessary columns
movies_cleaned = movies_data.drop(columns=['ONE-LINE', 'Gross', 'STARS'])

In [8]:
print(movies_cleaned.columns.values)

['MOVIES' 'YEAR' 'GENRE' 'RATING' 'VOTES' 'RunTime']


In [9]:
#Drop rows where 'GENRE' is missing since it is our target for classification
movies_cleaned = movies_cleaned.dropna(subset=['GENRE'])

In [10]:
# Remove non-numeric characters from 'YEAR' and keep only 4-digit years
movies_cleaned['YEAR'] = movies_cleaned['YEAR'].str.extract('(\d{4})')

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\khan\AppData\Local\Temp\ipykernel_4084\1551990934.py:2: SyntaxWarning: invalid escape sequence '\d'
  movies_cleaned['YEAR'] = movies_cleaned['YEAR'].str.extract('(\d{4})')


In [11]:
# Drop empty entries of vote's column
movies_cleaned= movies_cleaned.dropna(subset=['VOTES'])

In [12]:
# Remove unnecessary characters "," from Vote's entries
movies_cleaned['VOTES'] = movies_cleaned['VOTES'].str.replace(',', '').astype(float)

In [13]:
# Fill missing 'RATING' and 'RunTime' with median values
movies_cleaned['RATING'] = movies_cleaned['RATING'].fillna(movies_cleaned['RATING'].median())
movies_cleaned['RunTime'] = movies_cleaned['RunTime'].fillna(movies_cleaned['RunTime'].median())

# Check the cleaned dataset
print(movies_cleaned.head())


                                MOVIES  YEAR  \
0                        Blood Red Sky  2021   
1  Masters of the Universe: Revelation  2021   
2                     The Walking Dead  2010   
3                       Rick and Morty  2013   
5                          Outer Banks  2020   

                                        GENRE  RATING     VOTES  RunTime  
0      \nAction, Horror, Thriller                 6.1   21062.0    121.0  
1  \nAnimation, Action, Adventure                 5.0   17870.0     25.0  
2       \nDrama, Horror, Thriller                 8.2  885805.0     44.0  
3  \nAnimation, Adventure, Comedy                 9.2  414849.0     23.0  
5          \nAction, Crime, Drama                 7.6   25858.0     50.0  


In [43]:
movies_cleaned['GENRE'].value_counts()

GENRE
177    740
109    601
264    445
342    432
247    292
      ... 
185      1
61       1
469      1
462      1
267      1
Name: count, Length: 484, dtype: int64

In [14]:
movies_cleaned.head()

,MOVIES,YEAR,GENRE,RATING,VOTES,RunTime
0,Blood Red Sky,2021,"\nAction, Horror, Thriller",6.1,21062.0,121.0
1,Masters of the Universe: Revelation,2021,"\nAnimation, Action, Adventure",5.0,17870.0,25.0
2,The Walking Dead,2010,"\nDrama, Horror, Thriller",8.2,885805.0,44.0
3,Rick and Morty,2013,"\nAnimation, Adventure, Comedy",9.2,414849.0,23.0
5,Outer Banks,2020,"\nAction, Crime, Drama",7.6,25858.0,50.0


In [15]:
#X = movies_cleaned[['YEAR', 'RATING', 'VOTES', 'RunTime']]
# On the basis of all the coulumns, we are predicting y
#X= movies_cleaned.drop('GENRE', axis =1)
#y = movies_cleaned['GENRE']
X = movies_cleaned[['RATING', 'VOTES', 'RunTime']]
y = movies_cleaned['GENRE']

In [16]:
print(movies_cleaned.isnull().sum())

MOVIES     0
YEAR       0
GENRE      0
RATING     0
VOTES      0
RunTime    0
dtype: int64


In [17]:
lb = LabelEncoder()
for col in movies_cleaned.columns:
    if movies_cleaned[col].dtype == 'object' or movies_cleaned[col].dtype == 'category':
        movies_cleaned[col]=LabelEncoder().fit_transform(movies_cleaned[col])

In [18]:
label_encoder = LabelEncoder()
y_multiclass = label_encoder.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_multiclass, test_size=0.2, random_state=42)


In [20]:
movies_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8168 entries, 0 to 9979
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MOVIES   8168 non-null   int32  
 1   YEAR     8168 non-null   int32  
 2   GENRE    8168 non-null   int32  
 3   RATING   8168 non-null   float64
 4   VOTES    8168 non-null   float64
 5   RunTime  8168 non-null   float64
dtypes: float64(3), int32(3)
memory usage: 351.0 KB


In [21]:
print(X_train.dtypes)

RATING     float64
VOTES      float64
RunTime    float64
dtype: object


In [22]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test) #Scaling the test data too

# Training the Logistic Regression model
log_reg_multiclass = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=100)
log_reg_multiclass.fit(x_train_scaled, y_train)

LogisticRegression(multi_class='multinomial')

In [23]:
y_predict = log_reg_multiclass.predict(x_test_scaled) 

In [24]:
# Generate a classification report to see precision, recall, and F1-score
from sklearn.metrics import accuracy_score, classification_report
print("FILE LOADED | This operation took", time.time() - start, "seconds")

FILE LOADED | This operation took 43.939979553222656 seconds


In [47]:
"""import numpy as np
unique_classes = np.unique(y_test) 
#class_names = ['Action', 'Comedy', 'Drama', 'Thriller'] 
try:
    print("Classification Report:")
    report =classification_report(y_test, y_predict, labels=unique_classes)
  #  report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)
    print(report)
except Exception as e:
    print(f"An error occurred: {e}")"""

'import numpy as np\nunique_classes = np.unique(y_test) \n#class_names = [\'Action\', \'Comedy\', \'Drama\', \'Thriller\'] \ntry:\n    print("Classification Report:")\n    report =classification_report(y_test, y_predict, labels=unique_classes)\n  #  report = classification_report(y_test, y_pred, target_names=label_encoder.classes_, zero_division=0)\n    print(report)\nexcept Exception as e:\n    print(f"An error occurred: {e}")'

In [26]:
accuracy = accuracy_score(y_test, y_predict)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 14.08%


In [27]:
class_distribution = pd.Series(y_train).value_counts()
print(class_distribution)

177    595
109    476
264    355
342    332
247    224
      ... 
463      1
211      1
46       1
234      1
258      1
Name: count, Length: 448, dtype: int64


In [39]:
# movies_cleaned['GENRE'].value_counts()
# Get the genre counts and convert it to a DataFrame
genre_counts = movies_cleaned['GENRE'].value_counts().reset_index()

# Rename the columns to something more descriptive
genre_counts.columns = ['GENRE', 'COUNT']

print(genre_counts)


     GENRE  COUNT
0      177    740
1      109    601
2      264    445
3      342    432
4      247    292
..     ...    ...
479    185      1
480     61      1
481    469      1
482    462      1
483    267      1

[484 rows x 2 columns]


In [30]:
movies_cleaned.to_csv('movies_dataset_cleaned_data.csv', index=False)

In [31]:
from sklearn.neighbors import KNeighborsClassifier

In [33]:
knn.fit(X_train, y_train)

KNeighborsClassifier()

In [44]:
accuracy = accuracy_score(y_test, y_predict)
print(f'Accuracy: {accuracy:.2f}')
print(f"Accuracy: {accuracy * 100:.2f}%")



Accuracy: 0.14
Accuracy: 14.08%
